<a href="https://colab.research.google.com/github/atorgon/quantum_hackathon_2026/blob/master/Hackathon_quantique_sat_scheduling_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorithme quantique pour la planification de contacts satellite

Google Slides : https://docs.google.com/presentation/d/1efaTTJByQHJ7kTzVCp_8Mpq9VT9GhmQd7GlfnhfwqaM/edit?slide=id.p#slide=id.p

In [ ]:
!pip install qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 56.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 68.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit_aer]


## Lundi 26/10 Hackathon

Contraintes reformulées :
  1. Visibilités > 6 min
  2. Une station peut contacter un seul satellite à la fois
  3. Chaque satellite réalise AU MOINS un contact

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace
import qiskit.visualization as qviz
from qiskit_aer import AerSimulator
from qiskit.circuit.library import MCXGate
from collections import Counter
import math
from qiskit.circuit.library import QFTGate

In [ ]:
""" Le jeu de donnée initiale avec 2 solutions qui fonctionne """
vis_slots_min = [
    {"id": 0, "sat": "A", "station": "Toulouse", "start": 5, "end": 10},
    {"id": 1, "sat": "B", "station": "Toulouse", "start": 5, "end": 15},
    {"id": 2, "sat": "A", "station": "Kiruna",   "start": 0, "end": 10},
    {"id": 3, "sat": "C", "station": "Kiruna",   "start": 5, "end": 15},
    {"id": 4, "sat": "B", "station": "Toulouse", "start": 10, "end": 30},
    {"id": 5, "sat": "C", "station": "Toulouse", "start": 25, "end": 30},
    {"id": 6, "sat": "B", "station": "Kiruna",   "start": 30, "end": 40},
    {"id": 7, "sat": "A", "station": "Kiruna",   "start": 35, "end": 45}
]
n = len(vis_slots_min)

In [ ]:
""" Un autre jeu de donnée avec 2 solutions"""
vis_slots_min = [
    {"id": 0, "sat": "A", "station": "Toulouse", "start": 0, "end": 15},
    {"id": 1, "sat": "B", "station": "Toulouse", "start": 5, "end": 20},
    {"id": 2, "sat": "A", "station": "Kiruna", "start": 30, "end": 45},
    {"id": 3, "sat": "B", "station": "Kiruna", "start": 35, "end": 50},
    {"id": 4, "sat": "C", "station": "Kourou", "start": 25, "end": 35},
]
n = len(vis_slots_min)

In [ ]:
""" Le jeu de donnée initiale avec 2 solutions simplifié (en enlevant les slots inutilisable) """
vis_slots_min = [
    {"id": 0, "sat": "B", "station": "Toulouse", "start": 5, "end": 15},
    {"id": 1, "sat": "A", "station": "Kiruna",   "start": 0, "end": 10},
    {"id": 2, "sat": "C", "station": "Kiruna",   "start": 5, "end": 15},
    {"id": 3, "sat": "B", "station": "Toulouse", "start": 10, "end": 30},
    {"id": 4, "sat": "B", "station": "Kiruna",   "start": 30, "end": 40},
    {"id": 5, "sat": "A", "station": "Kiruna",   "start": 35, "end": 45}
]
n = len(vis_slots_min)

In [ ]:
""" Un jeu de donné avec 5 solutions"""
vis_slots_min = [
    {"id": 0, "sat": "A", "station": "Toulouse", "start": 0, "end": 8},
    {"id": 1, "sat": "B", "station": "Toulouse", "start": 9, "end": 16},
    {"id": 2, "sat": "C", "station": "Kiruna",   "start": 20, "end": 30},
    {"id": 3, "sat": "B", "station": "Kiruna",   "start": 20, "end": 26},
    {"id": 4, "sat": "D", "station": "Toulouse", "start": 35, "end": 45},
    {"id": 5, "sat": "C", "station": "Toulouse", "start": 50, "end": 60},
]

In [ ]:
def window_time (slot):
  return ( slot["end"] - slot["start"] )

In [ ]:
def apply_diffuser(circuit, nb_inputs):
    """
    Grover diffusion operator:
    inversion about the average.
    """
    n = nb_inputs
    circuit.h(range(n))
    circuit.x(range(n))

    circuit.h(n - 1)
    circuit.mcx(list(range(n - 1)), n - 1)
    circuit.h(n - 1)

    circuit.x(range(n))
    circuit.h(range(n))

In [ ]:
def overlap_stations(slot1, slot2):
  # True ssi slot1 et slot2 sont incompatibles à cause de condition 2
  if slot1["station"] == slot2["station"]:
    return not (slot1["end"] <= slot2["start"] or slot2["end"] <= slot1["start"])
  return False

In [ ]:
def number_overlap(slots):
    counter = 0
    n = len(slots)
    for i in range(n-1):
        for j in range(i+1, n):
            if overlap_stations(slots[i], slots[j]):
                counter += 1
    return counter

In [ ]:
# test sur nos données vis_slots_min
for i in range(n-1):
  for j in range(i+1, n):
    if overlap_stations(vis_slots_min[i], vis_slots_min[j]):
      print("overlap", i, j)
#print(number_qubit_ax(vis_slots_min))

overlap 0 1
overlap 1 4
overlap 2 3
overlap 4 5
overlap 6 7


In [ ]:
def duration(slot):
    return slot["end"] - slot["start"]

def flip_xq(circuit, slots, inputs):
  q_slots = []
  for i, slot in enumerate(slots):
    if duration(slot) < 6:
      circuit.x(inputs[i])
      q_slots.append(i)
  return q_slots

def condition_1(circuit, slots, inputs, c1):
  q_slots = flip_xq(circuit, slots, inputs)
  circuit.mcx(q_slots,c1[0])
  circuit.x(c1[0])
  flip_xq(circuit, slots, inputs)

def condition_1_inverse(circuit, slots, inputs, c1):
  q_slots = flip_xq(circuit, slots, inputs)
  circuit.x(c1[0])
  circuit.mcx(q_slots,c1[0])
  flip_xq(circuit, slots, inputs)

In [ ]:
def condition_2(circuit, slots, input_register, c2_register):
    n = len(slots)
    counter = 0
    for i in range(n-1):
        for j in range(i+1, n):
            if overlap_stations(slots[i], slots[j]):
                circuit.ccx(input_register[i], input_register[j], c2_register[counter])
                counter += 1

In [ ]:
def condition_3(circuit, slots, input_register, c3_register):
    condition_list = {}

    for item in slots:
        sat = item["sat"]
        condition_list.setdefault(sat, []).append(item["id"])

    for satellite, control_slots in condition_list.items():
        # X on all the input
        for slot in control_slots:
            circuit.x(input_register[slot])

        #target qubit: index of the satellite
        target = list(condition_list.keys()).index(satellite)
        #print("target:", target)
        #print("controls:", control_slots)

        # Mcx one target is the resgister of each satalite
        circuit.mcx([input_register[slot] for slot in control_slots],c3_register[target])

        #uncompute
        for slot in control_slots:
            circuit.x(input_register[slot])


In [ ]:
def merge_flags(circuit, slots, c1, c2, c3, out):
  ax_list = list(c1) + list(c2) + list(c3)
  for q in ax_list:
    circuit.x(q)
  #circuit.mcp(math.pi, ax_list, out[0])
  circuit.mcx(ax_list, out[0])
  for q in ax_list: # uncompute
    circuit.x(q)

In [ ]:
def apply_oracle(circuit, vis_slots_min, inputs, c1, c2, c3, out, exclude = []):
  condition_1(circuit, vis_slots_min, inputs, c1)
  condition_2(circuit, vis_slots_min, inputs, c2)
  condition_3(circuit, vis_slots_min, inputs, c3)
  merge_flags(circuit, vis_slots_min, c1, c2, c3, out)

  # uncomputing
  condition_3(circuit, vis_slots_min, inputs, c3)
  condition_2(circuit, vis_slots_min, inputs, c2)
  condition_1_inverse(circuit, vis_slots_min, inputs, c1)

  # On exclue les solutions dans la liste exclude
  for sol in exclude:
    for (i, bit) in enumerate(reversed(sol)):
      if bit == 0:
        circuit.x(inputs[i])
    circuit.mcx([q for q in inputs], out[0])
    #uncomputing
    for (i, bit) in enumerate(reversed(sol)):
      if bit == 0:
        circuit.x(inputs[i])


In [ ]:
def construct_circuit_grover(slots, nb_grover = -1, nb_sol = 1, exclude = []):
  n = len(slots)
  nb_sat = len(set([slot["sat"] for slot in slots]))
  nb_overlap = number_overlap(slots)

  # 5 registres
  inputs = QuantumRegister(n, "inputs")
  c1 = QuantumRegister(1, "c1")
  c2 = QuantumRegister(nb_overlap, "c2")
  c3 = QuantumRegister(nb_sat, "c3")
  out = QuantumRegister(1, "out")
  creg = ClassicalRegister(n, "classic") #classical registry

  circuit = QuantumCircuit(inputs, c1, c2, c3, out, creg)

  # Tour d'Hadamard
  for qubit in inputs:
    circuit.h(qubit)

  # Out initialisé à 1 puis H
  circuit.x(out)
  circuit.h(out)

  #Boucle d'oracle + diffusion
  if nb_grover == -1:
    nb_grover = round(math.pi / (4 * math.asin(math.sqrt(nb_sol/2**n))) - 1/2)
  print("Nombre d'étape de Grover = "+str(nb_grover)+" pour "+str(nb_sol)+" solutions")
  for _ in range(nb_grover):
    apply_oracle(circuit, slots, inputs, c1, c2, c3, out, exclude = exclude)
    apply_diffuser(circuit, n)

  return circuit, inputs, c1, c2, c3, out

In [ ]:
# Let's build and draw !
circuit, inputs, c1, c2, c3, out = construct_circuit_grover(vis_slots_min, nb_sol = 2)
circuit.draw(fold = 200)

Nombre d'étape de Grover = 8 pour 2 solutions


┌───┐┌───┐          ┌───┐     ┌───┐               ┌───┐┌───┐                         ┌───┐               ┌───┐                                   ┌───┐┌───┐┌───┐     ┌───┐┌───┐┌───┐     »
 inputs_0: ┤ H ├┤ X ├───────■──┤ X ├──■──┤ X ├────────────■──┤ X ├┤ X ├──────────────────────■──┤ X ├────────────■──┤ X ├────────────────────────────────■──┤ X ├┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├─────»
           ├───┤└───┘       │  └───┘  │  └───┘     ┌───┐  │  └───┘└───┘┌───┐┌───┐            │  └───┘┌───┐       │  └───┘          ┌───┐┌───┐            │  └───┘└───┘└───┘  │  ├───┤├───┤└───┘     »
 inputs_1: ┤ H ├────────────┼─────────■─────────■──┤ X ├──┼─────────■──┤ X ├┤ X ├────────────┼────■──┤ X ├───────■──────────────■──┤ H ├┤ X ├────────────┼───────────────────■──┤ X ├┤ H ├──────────»
           ├───┤            │  ┌───┐  │         │  └───┘  │  ┌───┐  │  ├───┤└───┘            │    │  ├───┤       │       ┌───┐  │  ├───┤└───┘            │                   │  ├───┤├───┤          »
 inputs_2: ┤ H ├───────■────┼──┤ X ├──┼─────────┼─────────■──┤ X ├──┼──┤ X ├─────────────────■────┼──┤ X ├───────┼────■──┤ H ├──┼──┤ X ├─────────────────┼───────────────────■──┤ X ├┤ H ├───────■──»
           ├───┤       │    │  ├───┤  │         │         │  └───┘  │  └───┘┌───┐┌───┐       │    │  └───┘┌───┐  │    │  ├───┤  │  ├───┤                 │                   │  ├───┤├───┤       │  »
 inputs_3: ┤ H ├───────■────┼──┤ X ├──┼─────────┼─────────┼─────────┼────■──┤ X ├┤ X ├───────┼────┼────■──┤ X ├──┼────■──┤ H ├──┼──┤ X ├─────────────────┼───────────────────■──┤ X ├┤ H ├───────■──»
           ├───┤       │    │  └───┘  │         │         │  ┌───┐  │    │  ├───┤├───┤       │    │    │  ├───┤  │    │  └───┘  │  └───┘     ┌───┐┌───┐  │                   │  ├───┤├───┤       │  »
 inputs_4: ┤ H ├───────┼────┼─────────┼─────────■────■────┼──┤ X ├──■────┼──┤ X ├┤ X ├───────┼────■────┼──┤ X ├──┼────┼─────────■─────────■──┤ H ├┤ X ├──┼───────────────────■──┤ X ├┤ H ├───────┼──»
           ├───┤┌───┐  │    │  ┌───┐  │         │    │    │  ├───┤  │    │  ├───┤├───┤       │    │    │  ├───┤  │    │         │         │  ├───┤└───┘  │  ┌───┐┌───┐┌───┐  │  ├───┤├───┤┌───┐  │  »
 inputs_5: ┤ H ├┤ X ├──┼────■──┤ X ├──┼─────────┼────■────┼──┤ X ├──┼────■──┤ X ├┤ X ├───────┼────┼────■──┤ X ├──┼────┼─────────┼─────────■──┤ X ├───────■──┤ X ├┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├──┼──»
           ├───┤└───┘  │    │  ├───┤  │         │    │    │  └───┘  │    │  ├───┤├───┤       │    │    │  ├───┤  │    │         │  ┌───┐  │  ├───┤       │  └───┘└───┘└───┘  │  ├───┤├───┤└───┘  │  »
 inputs_6: ┤ H ├──■────┼────┼──┤ X ├──┼─────────┼────┼────┼─────────■────┼──┤ X ├┤ X ├───────┼────■────┼──┤ X ├──┼────┼────■────┼──┤ H ├──┼──┤ X ├───────┼───────────────────■──┤ X ├┤ H ├───────┼──»
           ├───┤  │    │    │  ├───┤  │         │    │    │  ┌───┐  │    │  ├───┤└───┘       │    │    │  ├───┤  │    │    │    │  ├───┤  │  ├───┤┌───┐  │                 ┌─┴─┐├───┤├───┤┌───┐  │  »
 inputs_7: ┤ H ├──■────┼────┼──┤ X ├──┼─────────┼────┼────■──┤ X ├──┼────┼──┤ X ├────────────■────┼────┼──┤ X ├──┼────┼────■────┼──┤ H ├──┼──┤ X ├┤ H ├──┼─────────────────┤ X ├┤ H ├┤ X ├┤ H ├──┼──»
           └───┘  │    │  ┌─┴─┐├───┤  │  ┌───┐  │    │    │  └───┘  │    │  └───┘     ┌───┐  │    │    │  ├───┤  │    │    │    │  └───┘  │  └───┘└───┘┌─┴─┐               └───┘└───┘└───┘└───┘  │  »
       c1: ───────┼────┼──┤ X ├┤ X ├──┼──┤ X ├──┼────┼────┼─────────┼────┼─────────■──┤ X ├──┼────┼────┼──┤ X ├──┼────┼────┼────┼─────────┼────────────┤ X ├─────────────────────────────────────┼──»
                  │    │  └───┘└───┘┌─┴─┐├───┤  │    │    │         │    │         │  ├───┤  │    │    │  └───┘┌─┴─┐  │    │    │         │            └───┘                                     │  »
     c2_0: ───────┼────┼────────────┤ X ├┤ X ├──┼────┼────┼─────────┼────┼─────────■──┤ X ├──┼────┼────┼───────┤ X ├──┼────┼────┼─────────┼──────────────────────────────────────────────────────┼──»
                  │    │            └───┘└───┘┌─┴─┐

In [ ]:
circuit_oneiter, _,_,_,_,_ = construct_circuit(vis_slots_min, nb_sol = 2, nb_grover=-1)
circuit_oneiter.draw(fold=200)

NameError: name 'construct_circuit' is not defined

In [ ]:
""" On regarde le state vector en sortie du circuit """

state = Statevector.from_instruction(circuit)
input_indices = [circuit.find_bit(q).index for q in inputs]
probs = state.probabilities_dict(qargs=input_indices)
display(qviz.plot_distribution({
    bits: p for bits, p in probs.items() if p > 1e-10
}))


In [ ]:
# 4. Run the simulator
backend = AerSimulator()
circuit.measure(range(n), range(n))
tqc = transpile(circuit, backend)
result = backend.run(tqc, shots=4096).result()
counts = result.get_counts()

# printing most common results
print("Most common results :")
for bitstring, count in Counter(counts).most_common(20):
    # Qiskit prints classical bits reversed relative to register order.
    logical = bitstring[::-1]
    print(
        logical,
        "count:",
        count,
    )

# qviz histogram
logical = []
for bitstring in counts:
    # Qiskit prints classical bits reversed relative to register order.
    logical.append(bitstring[::-1])
data = dict(zip(logical,counts.values()))
qviz.plot_histogram(data, color='blue', title='Occurences des solutions (4096 shots)', number_to_keep=4, sort='asc')

In [ ]:
tqc.depth()

NameError: name 'tqc' is not defined

In [ ]:
def control_puissance_grover(p, slots, inputs, c1, c2, c3, out):
  circuit = QuantumCircuit(inputs, c1, c2, c3, out)
  for _ in range(p):
    apply_oracle(circuit, slots, inputs, c1, c2, c3, out)
    apply_diffuser(circuit, len(inputs))
  circuit.global_phase += p * math.pi
  return circuit.to_gate(label=f"G^{p}").control(1)

In [ ]:
""" Le circuit du Quantum Counting Algorithm parce qu'on est des ouf ! """
def construct_circuit_qca(slots, precision):
  n = len(slots)
  nb_sat = len(set([slot["sat"] for slot in slots]))
  nb_overlap = number_overlap(slots)

  # Les registres
  inputs = QuantumRegister(n, "inputs")
  c1 = QuantumRegister(1, "c1")
  c2 = QuantumRegister(nb_overlap, "c2")
  c3 = QuantumRegister(nb_sat, "c3")
  out = QuantumRegister(1, "out")
  creg = ClassicalRegister(precision, "classic") #classical registry
  prec = QuantumRegister(precision, "prec")

  circuit = QuantumCircuit(inputs, c1, c2, c3, out, creg, prec)

  # Tours d'Hadamard
  for qubit in inputs:
    circuit.h(qubit)
  for qubit in prec:
    circuit.h(qubit)

  # Out initialisé à 1 puis H
  circuit.x(out)
  circuit.h(out)

  #Les controled Grover
  work_qubits = (list(inputs) + list(c1) + list(c2) + list(c3) + list(out))
  for p in range(precision):
    control_gate = control_puissance_grover(2**p, slots, inputs, c1, c2, c3, out)
    circuit.append(control_gate, [prec[p]] + work_qubits)

  #La QFT inverse
  circuit.append(QFTGate(precision).inverse(), list(prec))

  #La mesure
  circuit.measure(prec, creg)

  return circuit

In [ ]:
def count_number_solution(slots, precision = 6, nbshots = 2048):
  n = len(slots)
  circuit = construct_circuit_qca(slots, precision)
  backend = AerSimulator()
  compiled = transpile(circuit, backend)
  counts = backend.run(compiled, shots=nbshots, seed_simulator=42).result().get_counts()

  #Le resultat le plus fréquent
  bits = max(counts, key=counts.get)
  y = int(bits, 2)
  theta = math.pi * y / (2**precision)
  M = round((2**n) * (math.sin(theta) ** 2))
  print("Les bits les plus fréquent : " + str(bits))
  print("y = " + str(y))
  print("theta = " + str(theta))
  print("M = " + str(M))

  #Check en sachant qu'il y a 2 solutions
  vrai_theta = math.asin(math.sqrt(2/(2**n)))
  print(f"Vrai theta = {vrai_theta} utilisation de la formule pour vérifier : {(2**n) * (math.sin(vrai_theta) ** 2)}")

  return M


In [ ]:
count_number_solution(vis_slots_min)

KeyboardInterrupt: 

In [ ]:
""" Let's draw le circuit du QCA (Quantum Counting Algorithm)"""
circuit = construct_circuit_qca(vis_slots_min, precision = 5)
circuit.draw(fold = 200)

┌───┐     ┌───────┐┌───────┐┌───────┐┌───────┐┌────────┐                          
 inputs_0: ┤ H ├─────┤0      ├┤0      ├┤0      ├┤0      ├┤0       ├──────────────────────────
           ├───┤     │       ││       ││       ││       ││        │                          
 inputs_1: ┤ H ├─────┤1      ├┤1      ├┤1      ├┤1      ├┤1       ├──────────────────────────
           ├───┤     │       ││       ││       ││       ││        │                          
 inputs_2: ┤ H ├─────┤2      ├┤2      ├┤2      ├┤2      ├┤2       ├──────────────────────────
           ├───┤     │       ││       ││       ││       ││        │                          
 inputs_3: ┤ H ├─────┤3      ├┤3      ├┤3      ├┤3      ├┤3       ├──────────────────────────
           ├───┤     │       ││       ││       ││       ││        │                          
 inputs_4: ┤ H ├─────┤4      ├┤4      ├┤4      ├┤4      ├┤4       ├──────────────────────────
           ├───┤     │       ││       ││       ││       ││        │                          
 inputs_5: ┤ H ├─────┤5      ├┤5      ├┤5      ├┤5      ├┤5       ├──────────────────────────
           └───┘     │       ││       ││       ││       ││        │                          
       c1: ──────────┤6  G^1 ├┤6  G^2 ├┤6  G^4 ├┤6  G^8 ├┤6  G^16 ├──────────────────────────
                     │       ││       ││       ││       ││        │                          
       c2: ──────────┤7      ├┤7      ├┤7      ├┤7      ├┤7       ├──────────────────────────
                     │       ││       ││       ││       ││        │                          
     c3_0: ──────────┤8      ├┤8      ├┤8      ├┤8      ├┤8       ├──────────────────────────
                     │       ││       ││       ││       ││        │                          
     c3_1: ──────────┤9      ├┤9      ├┤9      ├┤9      ├┤9       ├──────────────────────────
                     │       ││       ││       ││       ││        │                          
     c3_2: ──────────┤10     ├┤10     ├┤10     ├┤10     ├┤10      ├──────────────────────────
                     │       ││       ││       ││       ││        │                          
     c3_3: ──────────┤11     ├┤11     ├┤11     ├┤11     ├┤11      ├──────────────────────────
           ┌───┐┌───┐│       ││       ││       ││       ││        │                          
      out: ┤ X ├┤ H ├┤12     ├┤12     ├┤12     ├┤12     ├┤12      ├──────────────────────────
           ├───┤└───┘└───┬───┘└───┬───┘└───┬───┘└───┬───┘└───┬────┘┌─────────┐┌─┐            
   prec_0: ┤ H ├─────────■────────┼────────┼────────┼────────┼─────┤0        ├┤M├────────────
           ├───┤                  │        │        │        │     │         │└╥┘┌─┐         
   prec_1: ┤ H ├──────────────────■────────┼────────┼────────┼─────┤1        ├─╫─┤M├─────────
           ├───┤                           │        │        │     │         │ ║ └╥┘┌─┐      
   prec_2: ┤ H ├───────────────────────────■────────┼────────┼─────┤2 qft_dg ├─╫──╫─┤M├──────
           ├───┤                                    │        │     │         │ ║  ║ └╥┘┌─┐   
   prec_3: ┤ H ├────────────────────────────────────■────────┼─────┤3        ├─╫──╫──╫─┤M├───
           ├───┤                                             │     │         │ ║  ║  ║ └╥┘┌─┐
   prec_4: ┤ H ├─────────────────────────────────────────────■─────┤4        ├─╫──╫──╫──╫─┤M├
           └───┘                                                   └─────────┘ ║  ║  ║  ║ └╥┘
classic: 5/════════════════════════════════════════════════════════════════════╩══╩══╩══╩══╩═
                                                                               0  1  2  3  4

In [ ]:
# ============================================================
# Classical helper functions
# ============================================================

def minutes(t):
    return t

def duration(slot):
    return minutes(slot["end"]) - minutes(slot["start"])

def overlap(c1, c2):
    return (
        minutes(c1["start"]) < minutes(c2["end"])
        and minutes(c2["start"]) < minutes(c1["end"])
    )

def get_short_slots(slots, min_duration):
    return [
        c["id"] for c in slots
        if duration(c) < min_duration
    ]

def get_overlaps_by_satellite(slots):
    pairs = []

    for i, c1 in enumerate(slots):
        for c2 in slots[i+1:]:
            if c1["sat"] == c2["sat"] and overlap(c1, c2):
                pairs.append((c1["id"], c2["id"]))
    return pairs

def get_overlaps_by_station(slots):
    pairs = []

    for i, c1 in enumerate(slots):
        for c2 in slots[i+1:]:
            if c1["station"] == c2["station"] and overlap(c1, c2):
                pairs.append((c1["id"], c2["id"]))
    return pairs

def get_satellites(slots):
    return sorted(set(c["sat"] for c in slots))

def get_slots_for_satellite(slots, sat):
    return [
        c["id"] for c in slots
        if c["sat"] == sat
    ]

def is_solution(bitstring, slots, min_duration=6):
    selected = [
        c for c in slots
        if bitstring[c["id"]] == "1"
    ]

    # Too-short slots forbidden
    for c in selected:
        if duration(c) < min_duration:
            return False

    # Each satellite must appear at least once
    for sat in get_satellites(slots):
        if not any(c["sat"] == sat for c in selected):
            return False

    # Same satellite overlaps forbidden
    for i, c1 in enumerate(selected):
        for c2 in selected[i + 1:]:
            if c1["sat"] == c2["sat"] and overlap(c1, c2):
                return False

    # Same station overlaps forbidden
    for i, c1 in enumerate(selected):
        for c2 in selected[i + 1:]:
            if c1["station"] == c2["station"] and overlap(c1, c2):
                return False

    return True

In [ ]:
print(is_solution("001101" , vis_slots_min))
print(is_solution("101100" , vis_slots_min))
print(is_solution("101001" , vis_slots_min))

True
False
True


In [ ]:
def is_solution(bitstring, slots):
  return True

In [ ]:
def list_all_solution(slots):
  M = count_number_solution(slots, precision = 6, nbshots = 200)
  found_solutions = []
  while M > 0:
    circuit,_,_,_,_,_ = construct_circuit_grover(slots, nb_sol = M, exclude = found_solutions)
    # On run UN SEUL shot du circuit
    backend = AerSimulator()
    circuit.measure(range(n), range(n))
    tqc = transpile(circuit, backend)
    result = backend.run(tqc, shots=1, memory=True).result()
    solution = result.get_memory(tqc)[0][::-1]
    # On vérifie que c'est bien une solution
    if is_solution(solution, slots):
      print("We found "+str(solution)+" -> "+str(M-1)+" solution(s) left")
      found_solutions.append(solution)
      M -= 1
  return found_solutions

In [ ]:
def list_all_solution(input_dataset):
  M = count_number_solution(input_dataset, precision = 6, nbshots = 200)
  found_solutions = []
  while M > 0:
    circuit = construct_circuit_grover(input_dataset, nb_sol = M, exclude = found_solutions)
    # On run UN SEUL shot du circuit
    backend = AerSimulator()
    circuit.measure(range(n), range(n))
    tqc = transpile(circuit, backend)
    result = backend.run(tqc, shots=1, memory=True).result()
    solution = result.get_memory(tqc)[0][::-1]
    # On vérifie que c'est bien une solution
    if is_solution(solution, input_dataset):
      found_solutions.append(solution)
      M -= 1
  return found_solutions

In [ ]:
list_all_solution(vis_slots_min)

Les bits les plus fréquent : 111100
y = 60
theta = 2.945243112740431
M = 2
Vrai theta = 0.17771060084511175 utilisation de la formule pour vérifier : 2.0000000000000004
Nombre d'étape de Grover = 4 pour 2 solutions
We found 101001 -> 1 solution(s) left
Nombre d'étape de Grover = 6 pour 1 solutions
We found 000001 -> 0 solution(s) left


['101001', '000001']

# QAOA

$$x_i\in\{0,1\},\qquad i\in\{1,2,3,4,6,7\}\qquad$$ $$x_i=1 ≡ \text{un emplacements est chosi} $$

## condition 1
prétraitement, suppression de tous les emplacements dont la fenêtre est inférieure à 6
## condition 2
La vérification du chevauchement donne (1,4),(2,3),(6,7)
$$E_{cond2}=x_1x_4+x_2x_3+x_6x_7$$

##condition 3
Une station peut contacter un seul satellite à la fois
$$E_{cond3}=(1-x_2)(1-x_7)+(1-x_1)(1-x_4)(1-x_6)+(1-x_3)$$
$$E_{cond3}=3-x_1-x_2-x_3-x_4-x_6-x_7+x_2x_7+x_1x_4+x_1x_6+x_4x_6-x_1x_4x_6$$

##energy function
$$ E=  E_{cond2} + E_{cond3}$$
$$E=3-x_1-x_2-x_3-x_4-x_6-x_7+2x_1x_4+x_1x_6+x_4x_6+x_2x_7+x_2x_3+x_6x_7-x_1x_4x_6$$


## Ising representtion ( bit to spin)
$$x_i=\frac{1-s_i}{2}$$
Replacing that with E

$$E=\frac{13}{8}-\frac18 s_1+\frac14 s_3-\frac18 s_4-\frac18 s_6+\frac38 s_1s_4+\frac18 s_1s_6+\frac18 s_4s_6+\frac14 s_2s_3+\frac14 s_2s_7+\frac14 s_6s_7+\frac18 s_1s_4s_6$$

## cost Hamiltonian H_C
En supprimant la constante et chaque s est égal à Z

$$H_C=-\frac18 Z_0+\frac14 Z_2-\frac18 Z_3-\frac18 Z_4+\frac38 Z_0Z_3+\frac18 Z_0Z_4+\frac18 Z_3Z_4+\frac14 Z_1Z_2+\frac14 Z_1Z_5+\frac14 Z_4Z_5+\frac18 Z_0Z_3Z_4$$

Construire le circuit à partir de cette fonction

In [ ]:
def make_circuit(gamma, beta,P=1):
    qc = QuantumCircuit(6)

    for q in range(6):
        qc.h(q)

    for i in range(0,P):
      qc.rz(2 * gamma * (-1/8), 0)
      qc.rz(2 * gamma * ( 1/4), 2)
      qc.rz(2 * gamma * (-1/8), 3)
      qc.rz(2 * gamma * (-1/8), 4)

      qc.rzz(2 * gamma * (3/8), 0, 3)
      qc.rzz(2 * gamma * (1/8), 0, 4)
      qc.rzz(2 * gamma * (1/8), 3, 4)
      qc.rzz(2 * gamma * (1/4), 1, 2)
      qc.rzz(2 * gamma * (1/4), 1, 5)
      qc.rzz(2 * gamma * (1/4), 4, 5)

      ## ZZZ by hand
      qc.cx(0, 4)
      qc.cx(3, 4)
      qc.rz(2 * gamma * (1/8), 4)
      qc.cx(3, 4)
      qc.cx(0, 4)

      # mixer
      for q in range(6):
          qc.rx(2 * beta, q)

    return qc


In [ ]:
gamma = 1
beta = 2
qc=make_circuit(gamma,beta,P=1)
qc.measure_all()

qc.draw(fold=200)

┌───┐┌───────────┐                                                                                                   ┌───────┐ ░ ┌─┐               
   q_0: ┤ H ├┤ Rz(-0.25) ├─■────────────────────■────────────────────────────────────────────■────────────────────────────■──┤ Rx(4) ├─░─┤M├───────────────
        ├───┤└───────────┘ │                    │                    ┌───────┐               │                            │  └───────┘ ░ └╥┘┌─┐            
   q_1: ┤ H ├──────────────┼──────────■─────────┼──────────■─────────┤ Rx(4) ├───────────────┼────────────────────────────┼────────────░──╫─┤M├────────────
        ├───┤ ┌─────────┐  │          │ZZ(0.5)  │          │         ├───────┤               │                            │            ░  ║ └╥┘┌─┐         
   q_2: ┤ H ├─┤ Rz(0.5) ├──┼──────────■─────────┼──────────┼─────────┤ Rx(4) ├───────────────┼────────────────────────────┼────────────░──╫──╫─┤M├─────────
        ├───┤┌┴─────────┴┐ │ZZ(0.75)            │          │         └───────┘               │                            │  ┌───────┐ ░  ║  ║ └╥┘┌─┐      
   q_3: ┤ H ├┤ Rz(-0.25) ├─■────────────────────┼──────────┼─────────■───────────────────────┼──────■────────────────■────┼──┤ Rx(4) ├─░──╫──╫──╫─┤M├──────
        ├───┤├───────────┤                      │ZZ(0.25)  │         │ZZ(0.25)             ┌─┴─┐  ┌─┴─┐┌──────────┐┌─┴─┐┌─┴─┐├───────┤ ░  ║  ║  ║ └╥┘┌─┐   
   q_4: ┤ H ├┤ Rz(-0.25) ├──────────────────────■──────────┼─────────■──────────■──────────┤ X ├──┤ X ├┤ Rz(0.25) ├┤ X ├┤ X ├┤ Rx(4) ├─░──╫──╫──╫──╫─┤M├───
        ├───┤└───────────┘                                 │ZZ(0.5)             │ZZ(0.5) ┌─┴───┴─┐└───┘└──────────┘└───┘└───┘└───────┘ ░  ║  ║  ║  ║ └╥┘┌─┐
   q_5: ┤ H ├──────────────────────────────────────────────■────────────────────■────────┤ Rx(4) ├─────────────────────────────────────░──╫──╫──╫──╫──╫─┤M├
        └───┘                                                                            └───────┘                                     ░  ║  ║  ║  ║  ║ └╥┘
meas: 6/══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╩══╩══╩══╩══╩══╩═
                                                                                                                                          0  1  2  3  4  5

In [ ]:
def energy(bitstring_found):
    x1, x2, x3, x4, x6, x7 = [int(c) for c in bitstring_found]
    A = (1 - x2) * (1 - x7)
    B = (1 - x1) * (1 - x4) * (1 - x6)
    C = (1 - x3)
    overlap_cond = x1*x4 + x2*x3 + x6*x7
    return A + B + C + overlap_cond

In [ ]:
def run(gamma, beta):
    shotes=512
    qc = make_circuit(gamma, beta,P=1)
    qc.measure_all()
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=shotes).result().get_counts()

    total_energy = 0
    valid = 0
    for bitstring, count in counts.items():
        bitstring_found = bitstring[::-1]
        e = energy(bitstring_found)
        total_energy = total_energy + e * count
        if e == 0:
            valid = valid + count
    return total_energy /shotes, valid / shotes

import math
import numpy as np
gammas = np.linspace(0, 2*np.pi, 20)
betas  = np.linspace(0, 2*np.pi, 20)

best_energy = 1000
best_gamma = 0
best_beta = 0
table = []

for gamma in gammas:
    row = []
    for beta in betas:
        avg_e, p_valid = run(gamma, beta)
        row.append(p_valid)
        if avg_e < best_energy:
            best_energy = avg_e
            best_gamma = gamma
            best_beta = beta
    table.append(row)

print(" gamma =", best_gamma)
print("beta  =", best_beta)
print("lowest  energy =", best_energy)

 gamma = 4.6297154895007475
beta  = 0.3306939635357677
lowest  energy = 0.78271484375


In [ ]:
qc = make_circuit(best_gamma, best_beta,P=1)
qc.measure_all()
counts = backend.run(transpile(qc, backend), shots=2048).result().get_counts()
for bitstring, count in Counter(counts).most_common(5):
    solution = bitstring[::-1]
    print(solution, "count:", count, " energy:", energy(solution))

<class 'dict'>


In [ ]:
new_counts = {}
for bitstring, count in counts.items():
    s = bitstring[::-1]
    full = "0" + s[0:4] + "0" + s[4:6]
    new_counts[full] = count
import operator
solutions_final=sorted(new_counts.items(),key=operator.itemgetter(1),reverse=True)
print(solutions_final)

[('01010001', 276), ('00011001', 249), ('00101001', 100), ('00010011', 97), ('01100001', 90), ('00101010', 84), ('00100010', 80), ('01100000', 77), ('00011010', 74), ('01110000', 73), ('00110010', 72), ('01010010', 69), ('01100010', 69), ('00111000', 69), ('01110010', 69), ('00101000', 66), ('00111010', 62), ('00010010', 60), ('00001001', 37), ('01000001', 30), ('01110001', 24), ('01010011', 23), ('00011011', 22), ('00011000', 18), ('00111001', 18), ('00100011', 15), ('00000011', 13), ('01000011', 12), ('00001011', 12), ('01111000', 10), ('01010000', 9), ('00010000', 8), ('01011001', 7), ('01011010', 6), ('01101010', 6), ('00001000', 5), ('00111011', 4), ('00110000', 4), ('01101000', 3), ('00100000', 3), ('00110011', 3), ('00001010', 3), ('01101001', 3), ('01000000', 2), ('01011011', 2), ('00010001', 2), ('01111010', 2), ('00110001', 1), ('00000010', 1), ('01001001', 1), ('01000010', 1), ('01110011', 1), ('01001011', 1)]


In [ ]:
# printing most common results
print("Most common results:")
for bitstring, count in Counter(counts).most_common(5):
    # Qiskit prints classical bits reversed relative to register order.
    logical = bitstring[::-1]
    print(
        logical,
        "count:",
        count,
        "energy:",energy(logical)

    )

# qviz histogram
logical = []
for bitstring in counts:
    # Qiskit prints classical bits reversed relative to register order.
    logical.append(bitstring[::-1])
data = dict(zip(logical,counts.values()))
qviz.plot_histogram(data, color='blue', title='Scores of satellite contacts', number_to_keep=30, sort='asc')

In [ ]:
for bitstring, count in Counter(counts).most_common(200):
    solution = bitstring[::-1]
    print(solution, "count:", count, " energy:", energy(solution))

#QAOA
Reward for the logest duration

$$ E=λ E_{penality} - Σ d_i x_i $$
$$λ=Σ d_i +1$$
$$E= E_{penality} - Σ α x_i$$ , $$α=d_i/λ$$

$$d_i \space \text{la durée pour chaque entrée}$$
$$ E_{penality} \space \text{Notre dernier E, nous l'avons calculé avant}$$



In [ ]:

""" Le jeu de donnée initiale avec 2 solutions qui fonctionne """
vis_slots_min = [
    {"id": 0, "sat": "A", "station": "Toulouse", "start": 5, "end": 10},
    {"id": 1, "sat": "B", "station": "Toulouse", "start": 5, "end": 15},
    {"id": 2, "sat": "A", "station": "Kiruna",   "start": 0, "end": 10},
    {"id": 3, "sat": "C", "station": "Kiruna",   "start": 5, "end": 15},
    {"id": 4, "sat": "B", "station": "Toulouse", "start": 10, "end": 30},
    {"id": 5, "sat": "C", "station": "Toulouse", "start": 25, "end": 30},
    {"id": 6, "sat": "B", "station": "Kiruna",   "start": 30, "end": 40},
    {"id": 7, "sat": "A", "station": "Kiruna",   "start": 35, "end": 45}
]
n = len(vis_slots_min)

durations_all = []
total = 0
for slot in vis_slots_min:
    x = duration(slot)
    if x > 6:
        durations_all.append(x)
        total = total + x
lamda = total + 1
reward = [x / lamda for x in durations_all]

h_pen = [-1/8, 0, 1/4, -1/8, -1/8, 0]

def make_circuit(gamma, beta, P=1):
    qc = QuantumCircuit(6)
    for q in range(6):
        qc.h(q)

    for i in range(P):
        for q in range(6):
            qc.rz(2 * gamma * (h_pen[q] + reward[q] / 2), q)

        qc.rzz(2 * gamma * (3/8), 0, 3)
        qc.rzz(2 * gamma * (1/8), 0, 4)
        qc.rzz(2 * gamma * (1/8), 3, 4)
        qc.rzz(2 * gamma * (1/4), 1, 2)
        qc.rzz(2 * gamma * (1/4), 1, 5)
        qc.rzz(2 * gamma * (1/4), 4, 5)

        qc.cx(0, 4)
        qc.cx(3, 4)
        qc.rz(2 * gamma * (1/8), 4)
        qc.cx(3, 4)
        qc.cx(0, 4)

        for q in range(6):
            qc.rx(2 * beta, q)
    return qc

def penalty(bitstring_found):
    x1, x2, x3, x4, x6, x7 = [int(c) for c in bitstring_found]
    A = (1 - x2) * (1 - x7)
    B = (1 - x1) * (1 - x4) * (1 - x6)
    C = (1 - x3)
    overlap_cond = x1*x4 + x2*x3 + x6*x7
    return A + B + C + overlap_cond

def energy(bitstring_found):
    x1, x2, x3, x4, x6, x7 = [int(c) for c in bitstring_found]
    xs = [x1, x2, x3, x4, x6, x7]
    rewards = sum(r * x for r, x in zip(reward, xs))
    return penalty(bitstring_found) - rewards



shots = 2048

def run(gamma, beta):
    qc = make_circuit(gamma, beta, P=3)
    qc.measure_all()
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=shots).result().get_counts()

    total_energy = 0
    valid = 0
    for bitstring, count in counts.items():
        bitstring_found = bitstring[::-1]
        total_energy = total_energy + energy(bitstring_found) * count
        if penalty(bitstring_found) == 0:
            valid = valid + count
    return total_energy / shots, valid /shots

gammas = np.linspace(0, 2*np.pi, 20)
betas  = np.linspace(0, np.pi,   20)

best_energy = 1000
best_gamma = 0
best_beta = 0
table = []

for gamma in gammas:
    row = []
    for beta in betas:
        avg_e, p_valid = run(gamma, beta)
        row.append(p_valid)
        if avg_e < best_energy:
            best_energy = avg_e
            best_gamma = gamma
            best_beta = beta
    table.append(row)

print("gamma =", best_gamma)
print("beta  =", best_beta)
print("lowest energy =", best_energy)

qc = make_circuit(best_gamma, best_beta, P=3)
qc.measure_all()
counts = backend.run(transpile(qc, backend), shots=shots).result().get_counts()

for bitstring, count in Counter(counts).most_common(5):
    solution = bitstring[::-1]
    print(solution, "count:", count, " energy:", energy(solution, 3))

best_solution = None
best_duration = -1
for bitstring, count in counts.items():
    solution = bitstring[::-1]
    if penalty(solution) == 0:
        total_duration = sum(d * int(c) for d, c in zip(durations_all, solution))
        if total_duration > best_duration:
            best_duration = total_duration
            best_solution = solution
print("answer:", best_solution, " total duration:", best_duration)